# Health Info Chatbot (Open-Source LLM via Hugging Face)

A simple chatbot that answers general health questions using an open-source model (`mistralai/Mistral-7B-Instruct-v0.1`) from Hugging Face.

**Disclaimer:** This is an educational project. It is NOT a substitute for professional medical advice, diagnosis, or treatment. The chatbot includes a basic safety filter that redirects sensitive queries (e.g. self-harm) to professional help resources.

## Setup
You'll need a free Hugging Face account and access token: https://huggingface.co/settings/tokens

**Never hardcode your token in a notebook.** This notebook prompts for it securely at runtime using `getpass`, so it is never stored or committed to version control.

In [ ]:
!pip install -q transformers huggingface_hub

In [ ]:
# Securely log in to Hugging Face (token is not displayed or stored in the notebook)
from getpass import getpass
from huggingface_hub import login

hf_token = getpass("Enter your Hugging Face access token: ")
login(hf_token)

In [ ]:
from transformers import pipeline

chatbot = pipeline(
    "text-generation",
    model="mistralai/Mistral-7B-Instruct-v0.1",
    max_new_tokens=200
)

In [ ]:
# Safety filter: block sensitive self-harm related queries
UNSAFE_KEYWORDS = [
    "suicide",
    "kill myself",
    "overdose",
    "self harm",
    "how to die"
]

def is_safe(query: str) -> bool:
    query_lower = query.lower()
    return not any(word in query_lower for word in UNSAFE_KEYWORDS)

In [ ]:
def health_chatbot(query: str) -> str:
    """Answer a general health question using the Mistral model.

    Returns a safety message instead of a model response if the query
    matches a sensitive/self-harm related keyword.
    """
    if not is_safe(query):
        return "\u26a0\ufe0f Please contact a medical professional immediately."

    prompt = f"""Act like a friendly medical assistant.
Give general health information only.
Do NOT diagnose or prescribe medicine.
Always suggest consulting a doctor for serious issues.
Keep answers simple and clear.

Question: {query}
Answer:"""

    result = chatbot(prompt)
    generated_text = result[0]["generated_text"]

    # Strip the prompt from the model's output, keeping only the answer
    answer = generated_text.split("Answer:")[-1].strip()
    return answer

In [ ]:
# Try it out
query = input("Ask a health question: ")
print("\nChatbot:", health_chatbot(query))